# Baseline Model — Gradient-Boosted Trees (LightGBM)

Reference GBT round on the customer-level feature matrix
(`home_credit_risk.features.build_feature_matrix`). Goal: a benchmark **ROC-AUC** for
`TARGET` and a **null-aware feature shortlist** for the NN and tabular-transformer
rounds. Every run is logged to **MLflow** (`home-credit-risk` experiment) so the tuning
phase and the later model families sit on one leaderboard.

**Why there is no imputation step here.** The nulls in this matrix are almost always
*"no such history"*, not measurement gaps. That splits "imputation" into two very
different operations:

- **Structural encoding** (`modeling.apply_structural_encoding`) — a null `*_COUNT` /
  `*_SUM` / `*_RATE` / `HAS_*` means *zero* records, so its correct value is `0`. This is
  feature construction and is applied up front.
- **Statistical imputation** (median-filling behavioural `*_MEAN` / `*_MIN` / `*_MAX`) —
  the only *real* imputation. LightGBM **splits on nulls natively** (it learns a default
  direction), so trees need none of it. It is deferred to the NN round, where it must be
  fit **per fold** inside cross-validation to avoid leakage.

**Why selection happens here, on the trees.** Feature pruning uses **null-aware GBT gain
importance**, never missingness or median-filled correlations. Selecting on a
missingness-corrupted criterion is exactly the trap where a feature you drop might have
mattered after imputation — gain importance sidesteps it because the trees already use the
nulls. Only zero-variance and bit-identical columns are dropped up front.

**One thing the trees can't do for themselves:** express a quotient. `features.py` now
adds the affordability ratios (`RATIO_PAYMENT_RATE`, `RATIO_CREDIT_TO_INCOME`, …)
explicitly, and nulls the `365243` "never employed" sentinel in `DAYS_EMPLOYED` while
keeping its membership as `DAYS_EMPLOYED_ANOMALY`.

**Sections:** 1. Build & encode · 2. Prepare X/y · 3. 5-fold CV (ROC-AUC) · 4. Gain
importance & shortlist · 5. Persist shortlist

## 1. Build and structurally-encode the feature matrix

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.features import build_feature_matrix
from home_credit_risk.modeling import (
    apply_structural_encoding,
    category_mapping,
    constant_feature_columns,
    duplicate_feature_columns,
    split_xy,
    to_tree_matrix,
)
from home_credit_risk.tracking import (
    log_cv_metrics,
    log_feature_importance,
    log_text_artifact,
    resume_run,
    start_run,
)

pl.Config.set_tbl_rows(30)
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

N_SPLITS, SEED = 5, 42
PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=34,
    feature_fraction=0.7,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=1.0,
    min_child_samples=100,
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
)

In [ ]:
t0 = time.time()
app_train = pl.read_csv(datasets / "application_train.csv", infer_schema_length=10000)
fm = build_feature_matrix(app_train, datasets)
fm = apply_structural_encoding(fm)  # count/sum/rate/flag nulls -> 0 (not imputation)
print(f"matrix {fm.shape} built + encoded in {time.time() - t0:.1f}s")
print(f"row count preserved: {fm.height == app_train.height}")

## 2. Prepare X / y

`split_xy` drops `TARGET` and the `SK_ID*` identifiers. Two column filters follow, both
of which remove columns that carry no information rather than columns that merely look
weak:

- `constant_feature_columns` — zero-variance columns (e.g. `*_SK_DPD*_MIN_*`, always 0).
- `duplicate_feature_columns` — bit-identical columns. Redundant aggregations produce
  these: an inner `MIN` that is constant within every loan makes the outer mean/min/max
  coincide, and `aggregate_categorical`'s one-hot rate reproduces a hand-written domain
  rate exactly (`PREV_REFUSED_RATE` ≡ `PREV_NAME_CONTRACT_STATUS_Refused_RATE`). Trees
  don't care, but the NN round does, and dropping them here keeps the shortlist clean.

`category_mapping` + `to_tree_matrix` integer-code the driver's string categoricals from
an **explicit, sorted value→code mapping** and return their column indices for LightGBM's
native categorical handling. The mapping is fitted once and must be reused when scoring
`application_test`: encoding a second frame independently would assign codes by that
frame's own value order and silently remap categories.

In [ ]:
feature_cols, y = split_xy(fm)

constants = constant_feature_columns(fm, feature_cols)
feature_cols = [c for c in feature_cols if c not in set(constants)]

dupes = duplicate_feature_columns(fm, feature_cols)
feature_cols = [c for c in feature_cols if c not in set(dupes)]

# Fit the value->code mapping here and keep it: application_test must be encoded with
# this same mapping, not with its own value order.
categories = category_mapping(fm, feature_cols)
X, cat_idx = to_tree_matrix(fm, feature_cols, categories=categories)

print(f"features: {len(feature_cols)}")
print(f"  dropped {len(constants)} constant, {len(dupes)} duplicate")
print(f"categorical (integer-coded) columns: {len(cat_idx)}")
print(f"positive rate: {y.mean():.4f}  (class imbalance)")
print("new this round:", [c for c in feature_cols if c.startswith("RATIO_")])

## 3. Stratified 5-fold cross-validation

Out-of-fold ROC-AUC is the honest generalisation estimate. Nulls are fed to LightGBM
directly (no imputation). Gain importances are accumulated across folds for the selection
step.

The whole loop runs inside **one MLflow run** — per-fold AUCs are steps of a single
metric, so the run's headline number is the OOF score, the only one comparable across
models. `cv_seed` and `n_splits` are logged as parameters because a delta smaller than
the fold spread (±0.004 here) means nothing unless the folds were identical.

> **Early stopping is armed but has never fired.** Every fold reports `Did not meet early
> stopping`, with `best_iteration_` at 960-999 against `n_estimators=1000`: the tree
> budget ends each fit, not convergence. Raise the cap before tuning anything else —
> `best_iteration_` here is the argmax of validation AUC among the trees that were built,
> not a stopping point.
>
> **Known optimism.** `predict_proba` uses that `best_iteration_`, chosen on the same fold
> it is then scored against, so the OOF AUC is slightly flattered. The bias is small at
> `learning_rate=0.02` and identical across every run logged here, so run-to-run
> comparisons stay valid. Fix it before quoting an absolute number externally: nest a
> third split for the iteration choice, or fix `n_estimators` and predict with all trees.

In [ ]:
RUN_NAME = "gbt-fixes-and-ratios"

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(y))
importances = np.zeros(len(feature_cols))
fold_aucs, best_iters = [], []

with start_run(
    RUN_NAME,
    params={
        **PARAMS,
        "n_features": len(feature_cols),
        "n_constant_dropped": len(constants),
        "n_duplicate_dropped": len(dupes),
        "n_categorical": len(cat_idx),
        "cv_seed": SEED,
        "n_splits": N_SPLITS,
    },
    tags={"model": "lightgbm", "round": "gbt-baseline"},
) as run:
    run_id = run.info.run_id  # sections 4-5 reopen this run to attach the ranking

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        model = LGBMClassifier(**PARAMS)
        model.fit(
            X[tr],
            y[tr],
            eval_set=[(X[va], y[va])],
            eval_metric="auc",
            categorical_feature=cat_idx,
            callbacks=[early_stopping(50), log_evaluation(0)],
        )
        oof[va] = model.predict_proba(X[va])[:, 1]
        importances += model.booster_.feature_importance(importance_type="gain")
        auc = roc_auc_score(y[va], oof[va])
        fold_aucs.append(float(auc))
        best_iters.append(int(model.best_iteration_))
        print(f"fold {fold}: AUC {auc:.5f}  best_iter {model.best_iteration_}")

    oof_auc = float(roc_auc_score(y, oof))
    oof_pr = float(average_precision_score(y, oof))
    log_cv_metrics(
        fold_aucs,
        {
            "oof_roc_auc": oof_auc,
            "oof_pr_auc": oof_pr,
            "mean_best_iteration": float(np.mean(best_iters)),
        },
    )

print(f"\nOOF ROC-AUC: {oof_auc:.5f}")
print(f"OOF PR-AUC : {oof_pr:.5f}")
print(f"mean fold AUC: {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f}")

## 4. Gain importance and the selected shortlist

Ranking by gain (share of total split gain). This is the null-aware selection criterion:
the shortlist keeps features with meaningful gain and is what the NN round will consume.

In [ ]:
with resume_run(run_id):
    imp = log_feature_importance(feature_cols, importances.tolist())

nonzero = imp.filter(pl.col("gain") > 0).height
print(f"features with >0 gain: {nonzero} / {len(feature_cols)}")
imp.head(25)

In [ ]:
# The engineered features only (exclude the driver's own application columns)
app_cols = set(app_train.columns)
eng_top = imp.filter(~pl.col("feature").is_in(list(app_cols)))
print("Top engineered features by gain:")
eng_top.head(20)

In [ ]:
# Shortlist for the deferred NN round: keep features carrying real signal.
# Top-K by gain (K=200) is a pragmatic cut; every kept feature has >0 gain.
K = 200
shortlist = imp.head(K)["feature"].to_list()
print(
    f"shortlist size: {len(shortlist)}  "
    f"(cumulative gain kept: {imp.head(K)['gain_norm'].sum():.3f})"
)

## 5. Persist the shortlist and the ranking

Written next to the datasets (outside the repo, like the raw CSVs) **and** attached to
the MLflow run. The next rounds read `gbt_gain_importance.csv` / `gbt_shortlist.txt`:
the NN round runs on the shortlist with a per-fold impute + scale pipeline, and the
tabular-transformer round after it. Because the shortlist is an artifact of a specific
run, an AUC can always be traced back to the exact input set that produced it.

In [ ]:
out_dir = datasets.parent
imp.write_csv(out_dir / "gbt_gain_importance.csv")
(out_dir / "gbt_shortlist.txt").write_text("\n".join(shortlist))

# The shortlist also travels with the run, so a later round can reproduce exactly the
# input set a given AUC was obtained with.
with resume_run(run_id):
    log_text_artifact("\n".join(shortlist), "gbt_shortlist.txt")

print(f"wrote {out_dir / 'gbt_gain_importance.csv'}")
print(f"wrote {out_dir / 'gbt_shortlist.txt'}")
print(f"attached the shortlist to mlflow run {run_id}")